In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv("../scripts/sales.csv")
holidays = pd.read_csv("../scripts/holidays.csv")

In [3]:
sales["sale_date"] = pd.to_datetime(sales["sale_date"])
holidays["date"] = pd.to_datetime(holidays["date"])

sort data

In [4]:
sales = sales.sort_values(
    by=[
        "store_id",
        "product_id",
        "sale_date"
    ]
).reset_index(drop=True)

Calendar Features

In [5]:
sales["day_of_week"] = sales["sale_date"].dt.dayofweek

sales["day_name"] = sales["sale_date"].dt.day_name()

sales["month"] = sales["sale_date"].dt.month

sales["quarter"] = sales["sale_date"].dt.quarter

sales["year"] = sales["sale_date"].dt.year

sales["week_of_year"] = sales["sale_date"].dt.isocalendar().week.astype(int)

Weekend Flag

In [6]:
sales["weekend_flag"] = (
    sales["day_of_week"] >= 5
).astype(int)

Holiday Flag

In [7]:
holiday_dates = set(holidays["date"])

sales["holiday_flag"] = (
    sales["sale_date"].isin(holiday_dates)
).astype(int)

Lag Features

In [8]:
sales["lag_1"] = (
    sales.groupby(
        ["store_id", "product_id"]
    )["quantity_sold"]
    .shift(1)
)

In [9]:
sales["lag_7"] = (
    sales.groupby(
        ["store_id", "product_id"]
    )["quantity_sold"]
    .shift(7)
)

In [10]:
sales["lag_30"] = (
    sales.groupby(
        ["store_id", "product_id"]
    )["quantity_sold"]
    .shift(30)
)

Rolling Features

In [11]:
sales["rolling_7"] = (
    sales.groupby(
        ["store_id", "product_id"]
    )["quantity_sold"]
    .transform(
        lambda x: x.rolling(7).mean()
    )
)

In [12]:
sales["rolling_30"] = (
    sales.groupby(
        ["store_id", "product_id"]
    )["quantity_sold"]
    .transform(
        lambda x: x.rolling(30).mean()
    )
)

Revenue Per Unit

In [13]:
sales["revenue_per_unit"] = (
    sales["revenue"] /
    sales["quantity_sold"]
)

Fill Missing Values

In [14]:
sales = sales.fillna(0)

Check Dataset

In [15]:
sales.head()

,sale_date,store_id,product_id,quantity_sold,discount_percentage,revenue,day_of_week,day_name,month,quarter,year,week_of_year,weekend_flag,holiday_flag,lag_1,lag_7,lag_30,rolling_7,rolling_30,revenue_per_unit
0,2025-01-01,1,1,41,0,17652.55,2,Wednesday,1,1,2025,1,0,1,0.0,0.0,0.0,0.0,0.0,430.550000
1,2025-01-02,1,1,13,15,4757.58,3,Thursday,1,1,2025,1,0,0,41.0,0.0,0.0,0.0,0.0,365.967692
2,2025-01-03,1,1,10,0,4305.50,4,Friday,1,1,2025,1,0,0,13.0,0.0,0.0,0.0,0.0,430.550000
3,2025-01-04,1,1,27,15,9881.12,5,Saturday,1,1,2025,1,1,0,10.0,0.0,0.0,0.0,0.0,365.967407
4,2025-01-05,1,1,29,0,12485.95,6,Sunday,1,1,2025,1,1,0,27.0,0.0,0.0,0.0,0.0,430.550000


Check New Columns

In [16]:
sales.columns

Index(['sale_date', 'store_id', 'product_id', 'quantity_sold',
       'discount_percentage', 'revenue', 'day_of_week', 'day_name', 'month',
       'quarter', 'year', 'week_of_year', 'weekend_flag', 'holiday_flag',
       'lag_1', 'lag_7', 'lag_30', 'rolling_7', 'rolling_30',
       'revenue_per_unit'],
      dtype='str')

Save Engineered Dataset

In [17]:
sales.to_csv(
    "../scripts/featured_sales.csv",
    index=False
)

Verify Shape

In [18]:
print(sales.shape)

(730000, 20)


Feature Correlation

In [19]:
corr = sales[
    [
        "quantity_sold",
        "lag_1",
        "lag_7",
        "lag_30",
        "rolling_7",
        "rolling_30"
    ]
].corr()

corr

,quantity_sold,lag_1,lag_7,lag_30,rolling_7,rolling_30
quantity_sold,1.000000,0.049125,0.060509,0.016827,0.341127,0.103096
lag_1,0.049125,1.000000,0.059953,0.042063,0.364926,0.122037
lag_7,0.060509,0.059953,1.000000,0.094910,0.235667,0.232186
lag_30,0.016827,0.042063,0.094910,1.000000,0.186371,0.536734
rolling_7,0.341127,0.364926,0.235667,0.186371,1.000000,0.434963
rolling_30,0.103096,0.122037,0.232186,0.536734,0.434963,1.000000


Save Correlation

In [20]:
corr.to_csv("../reports/feature_correlation.csv")